# Vision Naturalistic DAM

DAM-on-naturalistic benchmark with direct `vit_base_patch16_224` vs `vit_base_patch16_clip_224.openai` comparison.

In [1]:
# This setup cell imports the DAM runner and summary helpers.
# OUTPUT_DIR points to the DAM results, while BASELINE_DIR points to the naturalistic baseline
# that DAM is compared against.

from pathlib import Path
import json
import pandas as pd

from vision.run_naturalistic_dam import run_naturalistic_dam
from vision.naturalistic_dam import summarize_dam_cross_category, summarize_encoder_head_to_head

DEVICE = "cpu"
BATCH_SIZE = 16
SEED = 0
BASELINE_DIR = Path("results/vision/naturalistic")
OUTPUT_DIR = Path("results/vision/naturalistic_dam")
INCLUDE_CLIP = True

## Fruits

In [2]:
# This cell runs the fruits DAM benchmark.
# The output summarizes the saved DAM rows for fruits and should be read relative to the
# naturalistic baseline: DAM is trying to improve retrieval without harming alignment too much.

fruits_summary = run_naturalistic_dam(
    category="fruits",
    device=DEVICE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    output_dir=OUTPUT_DIR,
    baseline_dir=BASELINE_DIR,
    dataset_pkl="datasets_peterson.pkl",
    image_root=".",
    include_clip=INCLUDE_CLIP,
)
fruits_summary

{'category': 'fruits',
 'models': ['vit_base_patch16_224::cls',
  'vit_base_patch16_224::mean_tokens',
  'vit_base_patch16_clip_224.openai::cls',
  'vit_base_patch16_clip_224.openai::mean_tokens'],
 'n_rows': 1032,
 'sanity_all_pass': False,
 'n_qualifying_wins': 0,
 'best_retrieval_gain': {'category': 'fruits',
  'split_label': 'fold_2',
  'split_seed': 2,
  'split_mode': 'balanced_exemplar_folds',
  'model_name': 'vit_base_patch16_clip_224.openai',
  'pooling': 'mean_tokens',
  'layer': 'layer_11',
  'anchor_kind': 'retrieval',
  'n_stored': 80,
  'n_probe': 40,
  'chance_accuracy': 1.25,
  'ident_accuracy_baseline': 100.0,
  'ident_avg_margin_baseline': 0.07056034882468623,
  'gen_accuracy_baseline': 37.5,
  'gen_avg_margin_baseline': -0.012203212100078192,
  'gen_avg_retrieved_human_similarity_baseline': 0.6376666675,
  'gen_avg_human_similarity_regret_baseline': 0.1771388875,
  'gen_same_concept_accuracy_baseline': 47.5,
  'human_rdm_spearman_baseline': 0.48125634610292334,
  'exa

## Vegetables

In [3]:
# This cell runs the vegetables DAM benchmark in the same format as fruits.
# The summary output is mainly a run artifact; the more interpretable comparisons appear in
# the later cross-category and encoder-comparison tables.

vegetables_summary = run_naturalistic_dam(
    category="vegetables",
    device=DEVICE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    output_dir=OUTPUT_DIR,
    baseline_dir=BASELINE_DIR,
    dataset_pkl="datasets_peterson.pkl",
    image_root=".",
    include_clip=INCLUDE_CLIP,
)
vegetables_summary

{'category': 'vegetables',
 'models': ['vit_base_patch16_224::cls',
  'vit_base_patch16_224::mean_tokens',
  'vit_base_patch16_clip_224.openai::cls',
  'vit_base_patch16_clip_224.openai::mean_tokens'],
 'n_rows': 864,
 'sanity_all_pass': False,
 'n_qualifying_wins': 8,
 'best_retrieval_gain': {'category': 'vegetables',
  'split_label': 'fold_2',
  'split_seed': 2,
  'split_mode': 'balanced_exemplar_folds',
  'model_name': 'vit_base_patch16_224',
  'pooling': 'mean_tokens',
  'layer': 'layer_6',
  'anchor_kind': 'alignment',
  'n_stored': 78,
  'n_probe': 39,
  'chance_accuracy': 1.2820512820512822,
  'ident_accuracy_baseline': 100.0,
  'ident_avg_margin_baseline': 0.16444022996178517,
  'gen_accuracy_baseline': 17.94871794871795,
  'gen_avg_margin_baseline': -0.04588698263029746,
  'gen_avg_retrieved_human_similarity_baseline': 0.4982051282051282,
  'gen_avg_human_similarity_regret_baseline': 0.3241025641025641,
  'gen_same_concept_accuracy_baseline': 33.33333333333333,
  'human_rdm_sp

## Animals

In [4]:
# This cell runs the animals DAM benchmark.
# Animals is the strongest positive DAM case in this project, so later tables should be read
# with special attention to whether animals gains are larger and cleaner than fruits/vegetables.

animals_summary = run_naturalistic_dam(
    category="animals",
    device=DEVICE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    output_dir=OUTPUT_DIR,
    baseline_dir=BASELINE_DIR,
    dataset_pkl="datasets_peterson.pkl",
    image_root=".",
    include_clip=INCLUDE_CLIP,
)
animals_summary

{'category': 'animals',
 'models': ['vit_base_patch16_224::cls',
  'vit_base_patch16_224::mean_tokens',
  'vit_base_patch16_clip_224.openai::cls',
  'vit_base_patch16_clip_224.openai::mean_tokens'],
 'n_rows': 3360,
 'sanity_all_pass': False,
 'n_qualifying_wins': 630,
 'best_retrieval_gain': {'category': 'animals',
  'split_label': 'stored_40_seed_3',
  'split_seed': 3,
  'split_mode': 'random_storage_curve',
  'model_name': 'vit_base_patch16_224',
  'pooling': 'cls',
  'layer': 'layer_11',
  'anchor_kind': 'retrieval',
  'n_stored': 40,
  'n_probe': 80,
  'chance_accuracy': 2.5,
  'ident_accuracy_baseline': 100.0,
  'ident_avg_margin_baseline': 0.06284979177730861,
  'gen_accuracy_baseline': 56.25,
  'gen_avg_margin_baseline': 0.007466354142619142,
  'gen_avg_retrieved_human_similarity_baseline': 0.7073750000000001,
  'gen_avg_human_similarity_regret_baseline': 0.14875,
  'gen_same_concept_accuracy_baseline': nan,
  'human_rdm_spearman_baseline': 0.10526401895764113,
  'exact_accurac

## Cross-Category Summary

In [5]:
# This cross-category DAM summary condenses the saved per-category CSVs into one object.
# Read it as the top-level answer to: where does DAM help, and where does it fail?
# Main finding from this notebook: fruits has no clean DAM win, vegetables has modest clean wins,
# and animals gives the strongest clean DAM improvement.

cross_summary = summarize_dam_cross_category(OUTPUT_DIR, ["fruits", "vegetables", "animals"])
cross_summary

{'categories': ['fruits', 'vegetables', 'animals'],
 'best_qualifying_win': {'fruits': None,
  'vegetables': {'category': 'vegetables',
   'split_label': 'fold_2',
   'split_seed': '2',
   'split_mode': 'balanced_exemplar_folds',
   'model_name': 'vit_base_patch16_224',
   'pooling': 'cls',
   'layer': 'layer_11',
   'anchor_kind': 'retrieval',
   'n_stored': '78',
   'n_probe': '39',
   'chance_accuracy': '1.2820512820512822',
   'ident_accuracy_baseline': '100.0',
   'ident_avg_margin_baseline': '0.056849945932699125',
   'gen_accuracy_baseline': '53.84615384615385',
   'gen_avg_margin_baseline': '-0.003939512681418282',
   'gen_avg_retrieved_human_similarity_baseline': '0.7235897435897435',
   'gen_avg_human_similarity_regret_baseline': '0.0987179487179487',
   'gen_same_concept_accuracy_baseline': '74.35897435897436',
   'human_rdm_spearman_baseline': '0.30382727361018574',
   'exact_accuracy_dam': '100.0',
   'clean_reextract_accuracy_dam': '100.0',
   'exact_avg_target_sim_dam': 

## ViT vs CLIP Summary

In [6]:
# This table is the matched ViT-vs-CLIP DAM comparison.
# Rows: matched branches with the same category, pooling choice, and anchor role.
# Anchor roles:
# - retrieval anchor: the layer chosen because it had the best baseline gen_accuracy
# - alignment anchor: the layer chosen because it had the best baseline human_rdm_spearman
# Important fields in the output rows:
# - baseline metrics: the pre-DAM reference values
# - *_delta fields: how much DAM changed retrieval, margin, regret, or alignment
# - classification / winner fields: whether ViT or CLIP gave the cleaner DAM result
# Main finding: ViT is currently the better DAM substrate, especially on animals.

head_to_head = summarize_encoder_head_to_head(OUTPUT_DIR, ["fruits", "vegetables", "animals"])
pd.DataFrame(head_to_head["comparison_rows"]).sort_values(["category", "pooling", "anchor_kind"])

,category,pooling,anchor_kind,vit_layer,clip_layer,outcome,vit_gen_accuracy_delta,clip_gen_accuracy_delta,vit_probe_rdm_delta,clip_probe_rdm_delta,vit_best_config,clip_best_config
0,animals,cls,retrieval,layer_11,layer_11,vit_clean_win,12.500000,7.500000,0.153553,-0.083653,"n=4,beta=0.2,alpha=0.1,steps=2","n=2,beta=0.2,alpha=0.1,steps=2"
1,animals,mean_tokens,alignment,layer_6,layer_6,retrieval_only_gain,5.000000,2.500000,-0.160997,-0.138041,"n=2,beta=0.2,alpha=0.05,steps=1","n=4,beta=0.05,alpha=0.05,steps=1"
2,animals,mean_tokens,retrieval,layer_11,layer_11,both_clean_wins,10.000000,2.500000,0.042233,0.013747,"n=4,beta=0.1,alpha=0.1,steps=1","n=4,beta=0.05,alpha=0.05,steps=2"
3,fruits,cls,retrieval,layer_11,layer_11,retrieval_and_alignment_gain,5.000000,7.500000,0.286555,-0.106968,"n=4,beta=0.2,alpha=0.1,steps=2","n=4,beta=0.5,alpha=0.05,steps=5"
4,fruits,mean_tokens,retrieval,layer_11,layer_11,retrieval_and_alignment_gain,5.000000,7.500000,0.158428,-0.152085,"n=2,beta=0.2,alpha=0.1,steps=2","n=2,beta=0.2,alpha=0.1,steps=2"
5,vegetables,cls,retrieval,layer_11,layer_11,vit_clean_win,2.564103,5.128205,0.136109,0.086520,"n=4,beta=0.05,alpha=0.05,steps=1","n=2,beta=0.2,alpha=0.1,steps=2"
6,vegetables,mean_tokens,retrieval,layer_11,layer_11,retrieval_and_alignment_gain,7.692308,7.692308,0.123568,0.063791,"n=2,beta=0.2,alpha=0.05,steps=2","n=2,beta=0.1,alpha=0.1,steps=2"


## Retrieval vs Alignment Frontier

In [7]:
# This table shows the retrieval-vs-alignment frontier extracted from the ViT-vs-CLIP comparison.
# Rows: Pareto-efficient DAM rows for each encoder/category/anchor branch.
# Read each row as a candidate tradeoff point:
# - higher gen_accuracy_delta is better for retrieval
# - higher probe_rdm_spearman_delta is better for preserving or improving human-like geometry
# - lower regret delta is better because it means less human-similarity loss
# Main finding: some branches, especially animals, support positive retrieval and positive alignment
# at the same time rather than forcing an immediate tradeoff.

pd.DataFrame(head_to_head["frontier_rows"]).sort_values(["category", "encoder", "anchor_kind"])

,category,pooling,anchor_kind,encoder,model_name,layer,qualifying_win,gen_accuracy_delta,gen_avg_margin_delta,gen_avg_human_similarity_regret_delta,probe_rdm_spearman_delta
3,animals,mean_tokens,alignment,clip,vit_base_patch16_clip_224.openai,layer_6,False,2.500000,-0.170930,-0.020000,-0.138041
1,animals,cls,retrieval,clip,vit_base_patch16_clip_224.openai,layer_11,False,7.500000,0.059285,-0.020750,-0.083653
5,animals,mean_tokens,retrieval,clip,vit_base_patch16_clip_224.openai,layer_11,True,2.500000,0.007381,-0.019125,0.013747
2,animals,mean_tokens,alignment,vit,vit_base_patch16_224,layer_6,False,5.000000,-0.170408,-0.033000,-0.160997
0,animals,cls,retrieval,vit,vit_base_patch16_224,layer_11,True,12.500000,0.120756,-0.070625,0.153553
4,animals,mean_tokens,retrieval,vit,vit_base_patch16_224,layer_11,True,10.000000,0.096714,-0.069000,0.042233
7,fruits,cls,retrieval,clip,vit_base_patch16_clip_224.openai,layer_11,False,7.500000,-0.029286,-0.020000,-0.106968
9,fruits,mean_tokens,retrieval,clip,vit_base_patch16_clip_224.openai,layer_11,False,7.500000,-0.064645,-0.012167,-0.152085
6,fruits,cls,retrieval,vit,vit_base_patch16_224,layer_11,False,5.000000,-0.092201,-0.034500,0.286555
8,fruits,mean_tokens,retrieval,vit,vit_base_patch16_224,layer_11,False,5.000000,-0.084462,-0.026611,0.158428


## Compact Delta Table

In [8]:
# This compact delta table is the easiest place to scan raw DAM effects across all categories.
# Rows: individual DAM configurations from the saved per-category CSVs.
# Fields:
# - anchor_kind: whether this row came from a retrieval-focused or alignment-focused baseline layer
# - dam_n / dam_beta / dam_alpha / dam_steps_multiplier: DAM hyperparameters
# - gen_accuracy_delta: change in generalization accuracy after DAM
# - gen_avg_margin_delta: change in separation from the best wrong item
# - gen_avg_human_similarity_regret_delta: change in human-similarity regret; lower is better
# - probe_rdm_spearman_delta: change in alignment of the probe geometry to human similarity
# - qualifying_win: whether the row met the clean-win criterion
# This is the raw table behind the cleaner summaries above.

rows = []
for category in ["fruits", "vegetables", "animals"]:
    csv_path = OUTPUT_DIR / f"{category}_combined.csv"
    if csv_path.exists():
        rows.append(pd.read_csv(csv_path))
combined = pd.concat(rows, ignore_index=True)
combined[[
    "category",
    "model_name",
    "pooling",
    "layer",
    "anchor_kind",
    "dam_n",
    "dam_beta",
    "dam_alpha",
    "dam_steps_multiplier",
    "gen_accuracy_delta",
    "gen_avg_margin_delta",
    "gen_avg_human_similarity_regret_delta",
    "probe_rdm_spearman_delta",
    "qualifying_win",
]].sort_values(["category", "gen_accuracy_delta"], ascending=[True, False]).head(30)

,category,model_name,pooling,layer,anchor_kind,dam_n,dam_beta,dam_alpha,dam_steps_multiplier,gen_accuracy_delta,gen_avg_margin_delta,gen_avg_human_similarity_regret_delta,probe_rdm_spearman_delta,qualifying_win
2189,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.05,0.10,2,12.50,0.116325,-0.070625,0.151192,True
2191,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.05,0.10,2,12.50,0.116318,-0.070625,0.151195,True
2197,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.10,0.10,2,12.50,0.116746,-0.070625,0.154907,True
2199,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.10,0.10,2,12.50,0.116732,-0.070625,0.154921,True
2201,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.20,0.05,2,12.50,0.117686,-0.065250,0.153860,True
2203,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.20,0.05,2,12.50,0.117680,-0.065250,0.153896,True
2204,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.20,0.10,1,12.50,0.118120,-0.065250,0.156412,True
2205,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.20,0.10,2,12.50,0.117440,-0.070625,0.161179,True
2206,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.20,0.10,1,12.50,0.118116,-0.065250,0.156468,True
2207,animals,vit_base_patch16_224,cls,layer_11,retrieval,2,0.20,0.10,2,12.50,0.117417,-0.070625,0.161210,True


## Animals Frontier

Focused DAM strength sweep on the validated branch: `animals / vit_base_patch16_224 / cls / layer_11`.


In [9]:
# This summary table is for the focused animals frontier analysis.
# It collapses the fixed branch `animals / vit_base_patch16_224 / cls / layer_11` into one row.
# Fields:
# - classification: the overall shape of the branch frontier
# - n_rows: number of evaluated DAM configurations for this branch
# - n_qualifying_wins: how many of those rows met the clean-win criterion
# - best_gen_accuracy_delta / best_probe_rdm_delta / best_margin_delta: best clean branch improvement
# Current finding: animals looks like a no_tradeoff branch in the validated pass.

frontier_summary_path = OUTPUT_DIR / "frontier" / "frontier_summary.json"
frontier_rows_path = OUTPUT_DIR / "frontier" / "frontier_rows.csv"
frontier_summary = json.loads(frontier_summary_path.read_text()) if frontier_summary_path.exists() else {"branches": {}}
frontier_rows = pd.read_csv(frontier_rows_path) if frontier_rows_path.exists() else pd.DataFrame()
animals_key = "animals::vit_base_patch16_224::cls::layer_11"
animals_frontier = frontier_summary.get("branches", {}).get(animals_key, {})
pd.DataFrame([{
    "branch": animals_key,
    "classification": animals_frontier.get("classification"),
    "n_rows": animals_frontier.get("n_rows"),
    "n_qualifying_wins": animals_frontier.get("n_qualifying_wins"),
    "best_gen_accuracy_delta": (animals_frontier.get("best_clean_win") or {}).get("gen_accuracy_delta"),
    "best_probe_rdm_delta": (animals_frontier.get("best_clean_win") or {}).get("probe_rdm_spearman_delta"),
    "best_margin_delta": (animals_frontier.get("best_clean_win") or {}).get("gen_avg_margin_delta"),
}])


,branch,classification,n_rows,n_qualifying_wins,best_gen_accuracy_delta,best_probe_rdm_delta,best_margin_delta
0,animals::vit_base_patch16_224::cls::layer_11,no_tradeoff,1800,844,12.5,0.159634,0.12437


In [10]:
# This table shows the alignment-preserving frontier for the focused animals branch.
# Rows: frontier configurations that are not dominated on the retrieval/alignment criteria.
# Fields:
# - split_label: which stored-set split the row came from
# - DAM hyperparameters: dam_n, dam_beta, dam_alpha, dam_steps_multiplier
# - gen_accuracy_delta / gen_avg_margin_delta: retrieval gains after DAM
# - gen_avg_human_similarity_regret_delta: change in regret; more negative is better
# - probe_rdm_spearman_delta: alignment change; positive means more human-like probe geometry
# - qualifying_win: whether the row passed the clean-win filter
# Main finding: useful animals DAM settings can improve retrieval and alignment together.

pd.DataFrame(animals_frontier.get("alignment_frontier", []))[[
    "split_label",
    "dam_n",
    "dam_beta",
    "dam_alpha",
    "dam_steps_multiplier",
    "gen_accuracy_delta",
    "gen_avg_margin_delta",
    "gen_avg_human_similarity_regret_delta",
    "probe_rdm_spearman_delta",
    "qualifying_win",
]].sort_values(["gen_accuracy_delta", "probe_rdm_spearman_delta"], ascending=[False, False]).head(20)


,split_label,dam_n,dam_beta,dam_alpha,dam_steps_multiplier,gen_accuracy_delta,gen_avg_margin_delta,gen_avg_human_similarity_regret_delta,probe_rdm_spearman_delta,qualifying_win
14,stored_40_seed_3,2,0.50,0.10,1,12.50,0.118805,-0.065250,0.164561,True
12,stored_40_seed_3,2,0.50,0.02,5,12.50,0.118144,-0.069875,0.162277,True
7,stored_40_seed_3,2,0.20,0.10,2,12.50,0.117440,-0.070625,0.161179,True
15,stored_40_seed_3,4,0.35,0.10,2,12.50,0.124370,-0.070750,0.159634,True
5,stored_40_seed_3,2,0.05,0.10,5,12.50,0.114900,-0.073375,0.150791,True
4,stored_40_seed_3,2,0.02,0.10,5,12.50,0.114296,-0.073500,0.145185,True
8,stored_40_seed_3,2,0.20,0.10,8,11.25,0.113859,-0.065250,0.179479,True
10,stored_40_seed_3,2,0.35,0.05,8,11.25,0.116639,-0.069875,0.178273,True
9,stored_40_seed_3,2,0.35,0.02,8,11.25,0.118174,-0.071000,0.163933,True
6,stored_40_seed_3,2,0.10,0.10,5,11.25,0.115601,-0.073375,0.159168,True


## Fruits Control Frontier

Same DAM strength sweep on the fragile control branch: `fruits / vit_base_patch16_224 / cls / layer_11`.


In [11]:
# This one-row summary is the control-branch version of the animals frontier summary.
# It fixes the branch to `fruits / vit_base_patch16_224 / cls / layer_11` and reports whether
# the frontier contains any clean DAM regime.
# Current finding: fruits is the fragile control branch and does not produce a clean DAM win.

fruits_key = "fruits::vit_base_patch16_224::cls::layer_11"
fruits_frontier = frontier_summary.get("branches", {}).get(fruits_key, {})
pd.DataFrame([{
    "branch": fruits_key,
    "classification": fruits_frontier.get("classification"),
    "n_rows": fruits_frontier.get("n_rows"),
    "n_qualifying_wins": fruits_frontier.get("n_qualifying_wins"),
    "best_gen_accuracy_delta": (fruits_frontier.get("best_clean_win") or {}).get("gen_accuracy_delta"),
    "best_probe_rdm_delta": (fruits_frontier.get("best_clean_win") or {}).get("probe_rdm_spearman_delta"),
    "best_margin_delta": (fruits_frontier.get("best_clean_win") or {}).get("gen_avg_margin_delta"),
}])


,branch,classification,n_rows,n_qualifying_wins,best_gen_accuracy_delta,best_probe_rdm_delta,best_margin_delta
0,fruits::vit_base_patch16_224::cls::layer_11,no_useful_dam_regime,540,0,None,None,None


In [12]:
# This table shows the retrieval-oriented frontier for the fruits control branch.
# Rows: the best fruits configurations if you only look at retrieval-oriented improvements.
# Read it together with qualifying_win and probe_rdm_spearman_delta:
# fruits can show retrieval gains, but those gains usually fail the full clean-win criterion
# because the margin or alignment behavior is not stable enough.

pd.DataFrame(fruits_frontier.get("retrieval_frontier", []))[[
    "split_label",
    "dam_n",
    "dam_beta",
    "dam_alpha",
    "dam_steps_multiplier",
    "gen_accuracy_delta",
    "gen_avg_margin_delta",
    "gen_avg_human_similarity_regret_delta",
    "probe_rdm_spearman_delta",
    "qualifying_win",
]].sort_values(["gen_accuracy_delta", "probe_rdm_spearman_delta"], ascending=[False, False]).head(20)


,split_label,dam_n,dam_beta,dam_alpha,dam_steps_multiplier,gen_accuracy_delta,gen_avg_margin_delta,gen_avg_human_similarity_regret_delta,probe_rdm_spearman_delta,qualifying_win
2,fold_0,2,0.20,0.10,8,10.0,-0.088452,-0.019250,0.292457,False
3,fold_0,2,0.50,0.10,5,10.0,-0.081992,-0.018194,0.259464,False
1,fold_0,2,0.20,0.05,8,5.0,-0.089852,-0.037500,0.303036,False
4,fold_0,4,0.02,0.10,8,5.0,-0.085423,-0.034250,0.288773,False
5,fold_0,4,0.10,0.10,3,5.0,-0.088510,-0.034500,0.287056,False
6,fold_2,4,0.02,0.02,1,5.0,-0.014850,-0.012556,0.014963,False
0,fold_0,2,0.02,0.10,3,2.5,-0.084263,-0.032500,0.286436,False


## Frontier Classification

Compact branch-level summary for the animals frontier and the fruits control.


In [13]:
# This final table compares the frontier classifications for all fixed branches included in
# the focused frontier analysis.
# Fields:
# - classification: overall branch shape such as no_tradeoff or no_useful_dam_regime
# - n_rows: number of tested configurations
# - n_qualifying_wins: number of rows that met the clean-win criterion
# Main finding: animals behaves like the strongest useful DAM branch, while fruits remains the
# failure control for this stage of the project.

pd.DataFrame([
    {"branch": key, **{k: v for k, v in value.items() if k in {"classification", "n_rows", "n_qualifying_wins"}}}
    for key, value in frontier_summary.get("branches", {}).items()
]).sort_values("branch")


,branch,classification,n_rows,n_qualifying_wins
0,animals::vit_base_patch16_224::cls::layer_11,no_tradeoff,1800,844
1,fruits::vit_base_patch16_224::cls::layer_11,no_useful_dam_regime,540,0


## Hard Identification DAM Test

This section tests a narrower hypothesis after breaking the naturalistic identification ceiling with large occlusion.\nThe question is: once identification is no longer trivial, does DAM still help more on `animals` than on `fruits`?\n

In [ ]:
# This cell loads the branch-level summary for the hard-identification DAM test.
# Each row is the best DAM configuration for one branch: animals or fruits,
# with or without small decision noise.
# Key fields:
# - ident_accuracy_baseline / gen_accuracy_baseline: baseline task accuracy under occlusion
# - ident_accuracy_dam / gen_accuracy_dam: the same metrics after DAM
# - *_delta: how much DAM changed the metric
# - probe_rdm_spearman_delta: whether DAM preserved or improved probe geometry
# - qualifying_hard_ident_win: True means the row satisfies the hard-ident clean-win criterion
# Main finding: animals stays a strong DAM-positive case after occlusion breaks the
# identification ceiling, while fruits still does not produce a clean win.

hard_ident_dir = ROOT / "results" / "vision" / "naturalistic_dam_hard_ident"
hard_ident_summary = json.loads((hard_ident_dir / "hard_ident_summary.json").read_text())
hard_ident_best = pd.DataFrame(hard_ident_summary["best_by_branch"]).T[
    [
        "category",
        "decision_noise_std",
        "ident_accuracy_baseline",
        "ident_accuracy_dam",
        "ident_accuracy_delta",
        "gen_accuracy_baseline",
        "gen_accuracy_dam",
        "gen_accuracy_delta",
        "gen_avg_margin_delta",
        "gen_avg_human_similarity_regret_delta",
        "probe_rdm_spearman_delta",
        "qualifying_hard_ident_win",
        "dam_n",
        "dam_beta",
        "dam_alpha",
        "dam_lmbda",
        "dam_steps_multiplier",
    ]
]
display(hard_ident_best)


In [ ]:
# This cell filters the full hard-identification DAM CSV down to the most useful rows.
# Each row is one split plus one DAM configuration under the fixed occlusion regime.
# The first table shows the strongest animals rows by DAM generalization gain.
# The second table shows the strongest fruits rows by DAM generalization gain.
# Read these tables as: which exact config gave the best improvement, and whether that
# improvement also kept margin and alignment healthy.
# Main finding: animals has many positive rows with improving margin, while fruits can
# gain accuracy but usually gives up margin quality, so it is still the weaker control.

hard_ident_rows = pd.read_csv(hard_ident_dir / "hard_ident_combined.csv")
cols = [
    "category", "decision_noise_std", "split_label", "ident_accuracy_baseline",
    "gen_accuracy_baseline", "gen_accuracy_dam", "gen_accuracy_delta",
    "gen_avg_margin_delta", "probe_rdm_spearman_delta",
    "qualifying_hard_ident_win", "dam_n", "dam_beta", "dam_alpha",
    "dam_lmbda", "dam_steps_multiplier"
]
animals_top = hard_ident_rows[hard_ident_rows["category"] == "animals"].sort_values(
    ["gen_accuracy_delta", "gen_avg_margin_delta"], ascending=[False, False]
)[cols].head(10)
fruits_top = hard_ident_rows[hard_ident_rows["category"] == "fruits"].sort_values(
    ["gen_accuracy_delta", "gen_avg_margin_delta"], ascending=[False, False]
)[cols].head(10)
display(animals_top)
display(fruits_top)


## Tradeoff Search

This section tests whether stronger DAM improves retrieval by overwriting the cue, or whether the useful regime is actually mild cue-preserving stabilization.\n

In [ ]:
# This cell loads the aggregated config summary from the capped hard-cue tradeoff sweep.
# Each row is one DAM configuration averaged across splits for a fixed branch.
# Key fields:
# - mean_gen_accuracy_delta: average retrieval gain from DAM
# - mean_gen_avg_margin_delta: average change in retrieval margin
# - mean_probe_rdm_spearman_delta: average change in probe geometry alignment
# - mean_cue_displacement: how far the recovered state moved away from the incoming cue
# - mean_target_pull_gain: how much the recovered state moved toward the ground-truth target
# Main finding: on animals, the best retrieval rows come from small cue displacement,
# so the strongest gains look more like gentle stabilization than aggressive overwriting.

tradeoff_dir = ROOT / "results" / "vision" / "naturalistic_dam_tradeoff_capped"
tradeoff_summary = json.loads((tradeoff_dir / "hard_ident_summary.json").read_text())
tradeoff_agg = pd.read_csv(tradeoff_dir / "hard_ident_aggregated.csv")
animals_tradeoff = tradeoff_agg[(tradeoff_agg["category"] == "animals") & (tradeoff_agg["decision_noise_std"] == 0.01)].sort_values(["mean_gen_accuracy_delta", "mean_gen_avg_margin_delta"], ascending=[False, False])
fruits_tradeoff = tradeoff_agg[(tradeoff_agg["category"] == "fruits") & (tradeoff_agg["decision_noise_std"] == 0.01)].sort_values(["mean_gen_accuracy_delta", "mean_gen_avg_margin_delta"], ascending=[False, False])
display(animals_tradeoff[["dam_n", "dam_beta", "dam_alpha", "dam_lmbda", "dam_steps_multiplier", "mean_gen_accuracy_delta", "mean_gen_avg_margin_delta", "mean_probe_rdm_spearman_delta", "mean_cue_displacement", "mean_target_pull_gain", "mean_qualifying_fraction"]].head(12))
display(fruits_tradeoff[["dam_n", "dam_beta", "dam_alpha", "dam_lmbda", "dam_steps_multiplier", "mean_gen_accuracy_delta", "mean_gen_avg_margin_delta", "mean_probe_rdm_spearman_delta", "mean_cue_displacement", "mean_target_pull_gain", "mean_qualifying_fraction"]].head(12))


In [ ]:
# This cell compares the highest-gain animal rows to the highest-displacement animal rows.
# The point is to test the tradeoff directly: if stronger completion were the source of the
# best gains, the highest-displacement rows should dominate. Instead, the top retrieval rows
# stay in a very low-displacement regime.

animals_top_gain = animals_tradeoff.sort_values("mean_gen_accuracy_delta", ascending=False)[["dam_n", "dam_beta", "dam_alpha", "dam_lmbda", "dam_steps_multiplier", "mean_gen_accuracy_delta", "mean_cue_displacement", "mean_target_pull_gain", "mean_probe_rdm_spearman_delta", "mean_gen_avg_margin_delta"]].head(10)
animals_top_displacement = animals_tradeoff.sort_values("mean_cue_displacement", ascending=False)[["dam_n", "dam_beta", "dam_alpha", "dam_lmbda", "dam_steps_multiplier", "mean_gen_accuracy_delta", "mean_cue_displacement", "mean_target_pull_gain", "mean_probe_rdm_spearman_delta", "mean_gen_avg_margin_delta"]].head(10)
display(animals_top_gain)
display(animals_top_displacement)


In [ ]:
# This cell shows the final hypothesis summary written by the runner.
# Read the `supported_hypothesis` field first. The current result says the tradeoff itself is
# weak, and the better-supported story is that useful DAM gains come from mild cue-preserving
# stabilization on animals rather than strong cue overwriting.

tradeoff_summary["hypothesis_summary"]
